<a href="https://colab.research.google.com/github/ntatfff/todo/blob/202411241258/ColabRadiomicsFeatureExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import radiomics
import numpy as np
import SimpleITK as sitk
import radiomics.featureextractor
import os
import six
from os import path
import pandas as pd

def featureExtractor(fileId):
  imagePath = './dataset/BraTS2021_Training_Data/%s/%s_flair.nii.gz' % (fileId, fileId)
  image = sitk.ReadImage(imagePath)
  maskPath ='./dataset/BraTS2021_Training_Data/%s/%s_kernel5_tumor.nii.gz' % (fileId, fileId)
  mask = sitk.ReadImage(maskPath)

  kernel = 5
  settings = {}
  settings['kernelRadius'] = kernel
  settings['maskedKernel'] = False
  settings['voxelBatch'] = 40000
  extractor = radiomics.featureextractor.RadiomicsFeatureExtractor(**settings)
  extractor.disableAllFeatures()
  extractor.enableFeatureClassByName('gldm')

  featureMap = extractor.execute(image, mask, voxelBased=True)

  for featureName, featureValue in six.iteritems(featureMap):
    if isinstance(featureValue, sitk.Image):
      fileFolder = './dataset/gldm/kernel5-radius5/tumor/%s' % (fileId)
      if path.exists(fileFolder) == False:
        os.mkdir(fileFolder)
      sitk.WriteImage(featureValue, '%s/%s.nrrd' % (fileFolder, featureName))
      print('Computed %s, stored as "%s/%s.nrrd"' % (featureName, fileFolder, featureName))
    # else:
    #   print('%s: %s' % (featureName, featureValue))

monitorFilePath = './dataset/gldm/kernel5-radius5/tumor.monitor.csv'
monitor = pd.read_csv(monitorFilePath, index_col='no')
for i, row in monitor.iterrows():
  if row['done'] != 0:
    continue
  fileId = row['file']
  print('Starting %s' % (fileId))
  featureExtractor(fileId)
  monitor.at[i, 'done'] = 1
  monitor.to_csv(monitorFilePath)

Starting BraTS2021_00140
Computed original_gldm_DependenceEntropy, stored as "./dataset/gldm/kernel5-radius5/tumor/BraTS2021_00140/original_gldm_DependenceEntropy.nrrd"
Computed original_gldm_DependenceNonUniformity, stored as "./dataset/gldm/kernel5-radius5/tumor/BraTS2021_00140/original_gldm_DependenceNonUniformity.nrrd"
Computed original_gldm_DependenceNonUniformityNormalized, stored as "./dataset/gldm/kernel5-radius5/tumor/BraTS2021_00140/original_gldm_DependenceNonUniformityNormalized.nrrd"
Computed original_gldm_DependenceVariance, stored as "./dataset/gldm/kernel5-radius5/tumor/BraTS2021_00140/original_gldm_DependenceVariance.nrrd"
Computed original_gldm_GrayLevelNonUniformity, stored as "./dataset/gldm/kernel5-radius5/tumor/BraTS2021_00140/original_gldm_GrayLevelNonUniformity.nrrd"
Computed original_gldm_GrayLevelVariance, stored as "./dataset/gldm/kernel5-radius5/tumor/BraTS2021_00140/original_gldm_GrayLevelVariance.nrrd"
Computed original_gldm_HighGrayLevelEmphasis, stored as